# AI Resume Optimizer

An intelligent resume optimization tool that tailors your resume for specific job descriptions using AI.

## How It Works
1. Run all cells in order
2. Mount Google Drive when prompted
3. Enter your Gemini API key
4. Upload or paste your resume
5. Paste a job description
6. Get an AI-optimized resume tailored to the job

## Features
- PDF resume upload or paste text
- AI-powered job description analysis
- Skill gap analysis with missing skills detection
- Strengths identification — see what you're already doing well
- Skill selection — choose which skills to emphasize
- Smart clarifying questions with suggested answers
- Resume optimization with ATS keyword matching
- Save optimized resumes by category (ML Engineer, Data Scientist, etc.)
- Optimization history timeline
- Resume library stored on Google Drive
- 4 tabs: Optimize, Library, History, Settings

In [ ]:
# Install dependencies
!pip install -q gradio google-genai PyMuPDF
!pip install pydantic==2.10.6
!pip install huggingface-hub==0.34.3

In [ ]:
import gradio as gr
from google import genai
import fitz  # PyMuPDF
import json
import os
import re
import time
from datetime import datetime
from typing import Dict, List, Optional

print("All dependencies installed and imported successfully!")

In [3]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Set up data directory
DRIVE_PATH = "/content/drive/MyDrive/AI_Resume_Optimizer"
os.makedirs(f"{DRIVE_PATH}/resumes", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/history", exist_ok=True)

print(f"Data directory: {DRIVE_PATH}")
print("Directories created successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data directory: /content/drive/MyDrive/AI_Resume_Optimizer
Directories created successfully!


In [4]:
# Configuration
GEMINI_MODEL = "gemini-2.0-flash"

CATEGORIES = [
    "ML Engineer",
    "Data Scientist",
    "AI Engineer",
    "Computer Vision",
    "Software Engineer",
    "Custom"
]

print(f"Model: {GEMINI_MODEL}")
print(f"Categories: {', '.join(CATEGORIES)}")
print("Configuration loaded!")

Model: gemini-2.0-flash
Categories: ML Engineer, Data Scientist, AI Engineer, Computer Vision, Software Engineer, Custom
Configuration loaded!


In [ ]:
# PDF Parsing and AI Engine

def parse_pdf(file_path: str) -> str:
    """Extract text from an uploaded PDF file."""
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    if not text.strip():
        raise ValueError("Could not extract any text from the PDF. The file may be empty, image-only, or corrupted.")
    return text.strip()


class ResumeAI:
    """Gemini-powered resume optimization engine."""

    def __init__(self, api_key: str):
        self.client = genai.Client(api_key=api_key)

    def _generate(self, prompt: str) -> str:
        """Send a prompt to Gemini and return the raw text response."""
        start = time.time()
        print(f"  [AI] Sending request to Gemini ({GEMINI_MODEL})...")
        response = self.client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )
        elapsed = time.time() - start
        print(f"  [AI] Response received in {elapsed:.1f}s")
        return response.text

    def _generate_json(self, prompt: str):
        """Generate content and parse as JSON with error handling."""
        raw = self._generate(prompt)
        text = raw.strip()
        text = re.sub(r'^```(?:json)?\s*\n?', '', text)
        text = re.sub(r'\n?```\s*$', '', text)
        text = text.strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"AI returned invalid JSON. This sometimes happens — please try again.\n"
                f"Parse error: {e}\n"
                f"Raw response (first 500 chars): {text[:500]}"
            )

    def analyze_job_description(self, jd_text: str) -> Dict:
        """Analyze a job description and extract structured requirements."""
        prompt = f"""You are an expert recruiter and job-description analyst.

Analyze the following job description and return a JSON object with exactly
these keys (no extra keys):

- "job_title": string - the title of the role
- "company": string - the company name (use "" if not mentioned)
- "must_have_skills": list of strings - required / mandatory skills
- "nice_to_have_skills": list of strings - preferred / bonus skills
- "key_responsibilities": list of strings - primary duties
- "keywords": list of strings - important keywords and phrases a resume
  should contain to rank well for this role
- "experience_level": string - e.g. "Entry", "Mid", "Senior", "Lead", etc.
- "summary": string - a 2-3 sentence plain-English summary of the role

Return ONLY valid JSON. No explanation, no markdown, no extra text.

--- JOB DESCRIPTION ---
{jd_text}
"""
        return self._generate_json(prompt)

    def generate_questions(self, resume_text: str, jd_analysis: Dict) -> Dict:
        """Compare resume against JD analysis and produce skill gaps, questions, and strengths.

        Returns a dict with keys: missing_skills, questions, strengths
        """
        jd_json = json.dumps(jd_analysis, indent=2)
        prompt = f"""You are a career coach helping someone tailor their resume to a specific job.

Below you will find:
1. The candidate's current resume text.
2. A structured analysis of the target job description.

Your task: analyze the gaps between the resume and the job requirements, then return a JSON object with exactly these three keys:

1. "missing_skills": an array of 3-6 skills/technologies the job requires but the resume does NOT clearly demonstrate. Each item has:
   - "skill": string - the skill or technology name
   - "relevance": string - one sentence explaining why this matters for the job

2. "questions": an array of 2-3 clarifying questions to ask the candidate. Each item has:
   - "id": integer starting at 1
   - "question": string - the question to ask
   - "why": string - brief explanation of why this matters
   - "suggested_answer": string - a plausible draft answer based on what you can infer from the resume (the candidate will edit this)

3. "strengths": an array of 2-3 strings, each a short sentence describing something the resume already does well for this job.

Return ONLY valid JSON. No explanation, no markdown, no extra text.

--- RESUME ---
{resume_text}

--- JOB ANALYSIS ---
{jd_json}
"""
        return self._generate_json(prompt)

    def optimize_resume(self, resume_text: str, jd_analysis: Dict, answers: List[Dict], selected_skills: Optional[List[str]] = None) -> str:
        """Generate an optimized resume tailored to the job description."""
        jd_json = json.dumps(jd_analysis, indent=2)
        answers_text = "\n".join(
            f"Q{a.get('id', '?')}: {a.get('question', '')}\nA: {a.get('answer', '')}"
            for a in answers
            if a.get('answer', '').strip()
        )

        skills_section = ""
        if selected_skills:
            skills_section = f"""
--- SKILLS TO HIGHLIGHT ---
The candidate wants these skills emphasized (they may have experience not shown in the resume):
{', '.join(selected_skills)}
"""

        prompt = f"""You are a professional resume writer with deep expertise in ATS
optimization and modern hiring practices.

Your task: rewrite the candidate's resume so it is **perfectly tailored**
to the target job description. Incorporate the additional information the
candidate provided in their answers to the clarifying questions.

Guidelines:
- Keep the resume **truthful** - do not fabricate experience.
- Use strong action verbs and quantify achievements where possible.
- Naturally weave in the important keywords from the job description.
- Use a clean, professional format (plain text, clear section headers).
- Aim for 1-2 pages of content.
- Place the most relevant experience and skills prominently.

Return ONLY the final resume text. No commentary, no markdown fences.

--- ORIGINAL RESUME ---
{resume_text}

--- JOB ANALYSIS ---
{jd_json}

--- CANDIDATE'S ADDITIONAL ANSWERS ---
{answers_text}
{skills_section}"""

        result = self._generate(prompt).strip()
        # Strip any markdown fences the model might add
        result = re.sub(r'^```(?:\w*)\s*\n?', '', result)
        result = re.sub(r'\n?```\s*$', '', result)
        return result.strip()


# Global AI engine instance (initialized when API key is set)
ai_engine = None
print("PDF parser and AI engine defined!")

In [6]:
# Resume Storage Manager

class ResumeStorage:
    """Manages saving and loading optimized resumes on Google Drive."""

    def __init__(self, base_path: str):
        self.base_path = base_path

    def save(self, category: str, optimized_text: str, job_title: str, company: str) -> str:
        """Save an optimized resume under a category folder."""
        cat_dir = os.path.join(self.base_path, "resumes", category.lower().replace(" ", "_"))
        os.makedirs(cat_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_title = job_title.replace(" ", "_")[:30]
        filename = f"{timestamp}_{safe_title}.txt"
        filepath = os.path.join(cat_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(optimized_text)

        # Append to history log
        history_path = os.path.join(self.base_path, "history", "log.json")
        history = []
        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                history = json.load(f)

        history.append({
            "timestamp": datetime.now().isoformat(),
            "category": category,
            "job_title": job_title,
            "company": company,
            "filename": filename
        })

        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        return filename

    def list_by_category(self) -> Dict:
        """List all saved resumes grouped by category."""
        resumes_dir = os.path.join(self.base_path, "resumes")
        categories = {}
        if not os.path.exists(resumes_dir):
            return categories

        for cat in sorted(os.listdir(resumes_dir)):
            cat_path = os.path.join(resumes_dir, cat)
            if os.path.isdir(cat_path):
                files = []
                for fname in sorted(os.listdir(cat_path), reverse=True):
                    fpath = os.path.join(cat_path, fname)
                    with open(fpath, "r", encoding="utf-8") as fh:
                        files.append({"name": fname, "content": fh.read()})
                categories[cat] = files

        return categories

    def get_history(self) -> List[Dict]:
        """Load optimization history."""
        history_path = os.path.join(self.base_path, "history", "log.json")
        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                return json.load(f)
        return []


storage = ResumeStorage(DRIVE_PATH)
print(f"Storage manager ready. Saving to: {DRIVE_PATH}")

Storage manager ready. Saving to: /content/drive/MyDrive/AI_Resume_Optimizer


In [7]:
# Application State

class AppState:
    """Manages current session state."""
    def __init__(self):
        self.resume_text = ""
        self.jd_analysis = None
        self.questions = []
        self.missing_skills = []    # [{skill, relevance}, ...]
        self.strengths = []          # [str, str, ...]
        self.selected_skills = []    # [str, ...] skills user toggled on

state = AppState()
print("Application state initialized!")

Application state initialized!


In [8]:
# Event Handlers

def on_set_api_key(api_key):
    """Initialize AI engine with API key."""
    global ai_engine
    if not api_key or not api_key.strip():
        return "Please enter a valid API key."
    try:
        ai_engine = ResumeAI(api_key.strip())
        return "API key set successfully! You can now optimize resumes."
    except Exception as e:
        return f"Error setting API key: {e}"

def on_upload_pdf(file):
    """Handle PDF file upload."""
    if file is None:
        return ""
    try:
        text = parse_pdf(file.name)
        state.resume_text = text
        return text
    except ValueError as e:
        return f"PDF error: {e}"
    except Exception as e:
        return f"Error parsing PDF: {e}"

def on_analyze_and_question(resume_text, jd_text):
    """Analyze JD and generate clarifying questions. Returns 12 values."""
    # Error returns — 12 gr.update() calls matching the UI outputs
    def error_return(msg):
        return (
            msg,                                              # analysis_output (Markdown value)
            gr.update(visible=False),                         # strengths_col
            gr.update(value=""),                              # strengths_md
            gr.update(visible=False),                         # skills_col
            gr.update(choices=[], value=[]),                  # skills_checkbox
            gr.update(visible=False),                         # questions_col
            gr.update(value=""), gr.update(value=""), gr.update(value=""),  # q1, q2, q3 labels
            gr.update(value=""), gr.update(value=""), gr.update(value=""),  # a1, a2, a3
        )

    if not ai_engine:
        return error_return("Please set your API key first.")
    if not resume_text.strip():
        return error_return("Please enter or upload your resume first.")
    if not jd_text.strip():
        return error_return("Please enter a job description.")

    state.resume_text = resume_text

    try:
        # Step 1: Analyze JD
        analysis = ai_engine.analyze_job_description(jd_text)
        state.jd_analysis = analysis

        # Format analysis display
        skills = ', '.join(analysis.get('must_have_skills', []))
        nice = ', '.join(analysis.get('nice_to_have_skills', []))
        resps = '\n'.join([f"- {r}" for r in analysis.get('key_responsibilities', [])])

        analysis_md = f"""### Job Analysis Complete!

**Position:** {analysis.get('job_title', 'N/A')}
**Company:** {analysis.get('company', 'N/A')}
**Level:** {analysis.get('experience_level', 'N/A')}

**Must-Have Skills:** {skills}

**Nice-to-Have:** {nice}

**Key Responsibilities:**
{resps}

**Summary:** {analysis.get('summary', '')}
"""

        # Step 2: Generate questions + skill gaps + strengths
        result = ai_engine.generate_questions(resume_text, analysis)
        questions = result.get("questions", [])
        missing_skills = result.get("missing_skills", [])
        strengths = result.get("strengths", [])

        state.questions = questions
        state.missing_skills = missing_skills
        state.strengths = strengths

        # Format strengths as markdown bullet list
        if isinstance(strengths, list) and strengths:
            strengths_text = "\n".join([f"- {s}" for s in strengths])
        else:
            strengths_text = ""

        # Build skill choices from missing_skills
        skill_choices = []
        if isinstance(missing_skills, list):
            for s in missing_skills:
                if isinstance(s, dict) and "skill" in s:
                    skill_choices.append(s["skill"])

        # Extract question labels and suggested answers
        q1 = questions[0] if len(questions) > 0 else {}
        q2 = questions[1] if len(questions) > 1 else {}
        q3 = questions[2] if len(questions) > 2 else {}

        def fmt_q(q):
            if not q or not isinstance(q, dict):
                return ""
            text = f"**{q.get('question', '')}**"
            if q.get('why'):
                text += f"\n\n*Why this matters: {q['why']}*"
            return text

        def get_suggested(q):
            if isinstance(q, dict):
                return q.get("suggested_answer", "")
            return ""

        return (
            analysis_md,                                                    # analysis_output
            gr.update(visible=bool(strengths_text)),                        # strengths_col
            gr.update(value=strengths_text),                                # strengths_md
            gr.update(visible=bool(skill_choices)),                         # skills_col
            gr.update(choices=skill_choices, value=skill_choices),           # skills_checkbox
            gr.update(visible=bool(questions)),                             # questions_col
            gr.update(value=fmt_q(q1)),                                     # q1_label
            gr.update(value=fmt_q(q2)),                                     # q2_label
            gr.update(value=fmt_q(q3)),                                     # q3_label
            gr.update(value=get_suggested(q1)),                             # answer1
            gr.update(value=get_suggested(q2)),                             # answer2
            gr.update(value=get_suggested(q3)),                             # answer3
        )
    except Exception as e:
        return error_return(f"Error during analysis: {e}")

def on_optimize(resume_text, skills_checkbox, a1, a2, a3):
    """Generate optimized resume."""
    if not ai_engine:
        return "Please set your API key first."
    if not state.jd_analysis:
        return "Please analyze a job description first."

    state.resume_text = resume_text
    state.selected_skills = list(skills_checkbox) if isinstance(skills_checkbox, (list, tuple)) else []

    answers = []
    for i, ans in enumerate([a1, a2, a3]):
        if ans and ans.strip() and i < len(state.questions):
            answers.append({
                "id": state.questions[i].get("id", i + 1),
                "question": state.questions[i]["question"],
                "answer": ans.strip()
            })

    try:
        optimized = ai_engine.optimize_resume(
            state.resume_text,
            state.jd_analysis,
            answers,
            selected_skills=state.selected_skills
        )
        return optimized
    except Exception as e:
        return f"Error optimizing resume: {e}"

def on_save(category, optimized_text):
    """Save the optimized resume."""
    if not optimized_text or not optimized_text.strip():
        return "No optimized resume to save. Run optimization first."
    if not category:
        return "Please select a category."

    job_title = state.jd_analysis.get("job_title", "unknown") if state.jd_analysis else "unknown"
    company = state.jd_analysis.get("company", "unknown") if state.jd_analysis else "unknown"

    try:
        filename = storage.save(category, optimized_text, job_title, company)
        return f"Saved successfully as: {filename} (Category: {category})"
    except Exception as e:
        return f"Error saving: {e}"

def on_load_library(selected_category):
    """Load and display saved resumes, optionally filtered by category."""
    try:
        categories = storage.list_by_category()
        if not categories:
            return "No saved resumes yet. Optimize and save a resume first!"

        md = ""
        for cat, files in categories.items():
            cat_display = cat.replace('_', ' ').title()
            # Filter by category when not "All"
            if selected_category and selected_category != "All":
                if cat_display.lower() != selected_category.lower():
                    continue

            md += f"\n## {cat_display} ({len(files)} resumes)\n"
            for f in files[:5]:  # Show latest 5 per category
                md += f"\n### {f['name']}\n"
                preview = f['content'][:600]
                md += f"```\n{preview}\n{'...' if len(f['content']) > 600 else ''}\n```\n"

        if not md.strip():
            return f"No resumes found for category: {selected_category}"
        return md
    except Exception as e:
        return f"Error loading library: {e}"

def on_load_history():
    """Load and display optimization history grouped by date."""
    try:
        history = storage.get_history()
        if not history:
            return "No optimization history yet. Optimize and save a resume to start tracking!"

        # Group entries by date
        from collections import defaultdict
        grouped = defaultdict(list)
        today = datetime.now().date()

        for entry in reversed(history):  # Most recent first
            try:
                ts = datetime.fromisoformat(entry["timestamp"])
                entry_date = ts.date()
                if entry_date == today:
                    date_label = "Today"
                elif entry_date == today.replace(day=today.day - 1) if today.day > 1 else today:
                    date_label = "Yesterday"
                else:
                    date_label = entry_date.strftime("%B %d, %Y")
                grouped[date_label].append(entry)
            except (KeyError, ValueError):
                continue

        md = "## Optimization History\n"
        for date_label, entries in grouped.items():
            md += f"\n### {date_label}\n"
            for entry in entries:
                job = entry.get("job_title", "Unknown")
                company = entry.get("company", "")
                cat = entry.get("category", "")
                company_str = f" at **{company}**" if company else ""
                cat_str = f" ({cat})" if cat else ""
                md += f"- **{job}**{company_str}{cat_str}\n"

        return md
    except Exception as e:
        return f"Error loading history: {e}"

print("Event handlers defined!")

Event handlers defined!


In [9]:
# Gradio UI

def build_app():
    with gr.Blocks(
        title="AI Resume Optimizer",
        theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="slate", neutral_hue="slate")
    ) as app:

        gr.Markdown("# AI Resume Optimizer\nTailor your resume for any job description using AI")

        with gr.Tabs():
            # === TAB 1: OPTIMIZE ===
            with gr.Tab("Optimize Resume"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### Step 1: Your Resume")
                        pdf_upload = gr.File(label="Upload PDF (optional)", file_types=[".pdf"])
                        resume_input = gr.Textbox(
                            label="Resume Text",
                            placeholder="Upload a PDF above or paste your resume here...",
                            lines=12
                        )
                    with gr.Column(scale=1):
                        gr.Markdown("### Step 2: Job Description")
                        jd_input = gr.Textbox(
                            label="Job Description",
                            placeholder="Paste the complete job description here...",
                            lines=14
                        )

                analyze_btn = gr.Button("Analyze Job Description", variant="primary", size="lg")

                # Analysis results
                analysis_output = gr.Markdown("")

                # Strengths section (hidden until analysis runs)
                with gr.Column(visible=False) as strengths_col:
                    gr.Markdown("### What You're Already Doing Well")
                    strengths_md = gr.Markdown("")

                # Skills section (hidden until analysis runs)
                with gr.Column(visible=False) as skills_col:
                    gr.Markdown("### Skills to Highlight")
                    skills_checkbox = gr.CheckboxGroup(
                        choices=[],
                        label="Select skills to emphasize in your optimized resume"
                    )

                # Questions section (hidden until analysis runs)
                with gr.Column(visible=False) as questions_col:
                    gr.Markdown("### Answer These Questions to Personalize Your Resume")
                    q1_label = gr.Markdown("")
                    answer1 = gr.Textbox(label="Your Answer", lines=2)
                    q2_label = gr.Markdown("")
                    answer2 = gr.Textbox(label="Your Answer", lines=2)
                    q3_label = gr.Markdown("")
                    answer3 = gr.Textbox(label="Your Answer", lines=2)
                    optimize_btn = gr.Button("Optimize My Resume", variant="primary", size="lg")

                # Optimized result
                gr.Markdown("### Optimized Resume")
                optimized_output = gr.Textbox(label="Optimized Resume", lines=15, interactive=True)

                # Save section
                with gr.Row():
                    category_dropdown = gr.Dropdown(
                        choices=CATEGORIES,
                        label="Save to Category",
                        scale=2
                    )
                    save_btn = gr.Button("Save Resume", variant="secondary", scale=1)
                save_status = gr.Textbox(label="Save Status", interactive=False)

            # === TAB 2: LIBRARY ===
            with gr.Tab("Resume Library"):
                library_filter = gr.Dropdown(
                    choices=["All"] + CATEGORIES,
                    value="All",
                    label="Filter by Category"
                )
                refresh_btn = gr.Button("Load Saved Resumes", variant="primary")
                library_output = gr.Markdown("Click 'Load Saved Resumes' to view your resume library.")

            # === TAB 3: HISTORY ===
            with gr.Tab("History"):
                history_btn = gr.Button("Load History", variant="primary")
                history_output = gr.Markdown("Click 'Load History' to view your optimization timeline.")

            # === TAB 4: SETTINGS ===
            with gr.Tab("Settings"):
                gr.Markdown("### API Configuration")
                api_key_input = gr.Textbox(
                    label="Gemini API Key",
                    type="password",
                    placeholder="Enter your Gemini API key..."
                )
                api_key_btn = gr.Button("Set API Key", variant="primary")
                api_status = gr.Textbox(label="Status", interactive=False, value="API key not set")

                gr.Markdown(f"### Data Location\n`{DRIVE_PATH}`\n\nYour resumes are saved to Google Drive automatically.")

        # === EVENT WIRING ===
        api_key_btn.click(fn=on_set_api_key, inputs=[api_key_input], outputs=[api_status])

        pdf_upload.change(fn=on_upload_pdf, inputs=[pdf_upload], outputs=[resume_input])

        analyze_btn.click(
            fn=on_analyze_and_question,
            inputs=[resume_input, jd_input],
            outputs=[
                analysis_output,
                strengths_col, strengths_md,
                skills_col, skills_checkbox,
                questions_col,
                q1_label, q2_label, q3_label,
                answer1, answer2, answer3
            ]
        )

        optimize_btn.click(
            fn=on_optimize,
            inputs=[resume_input, skills_checkbox, answer1, answer2, answer3],
            outputs=[optimized_output]
        )

        save_btn.click(
            fn=on_save,
            inputs=[category_dropdown, optimized_output],
            outputs=[save_status]
        )

        refresh_btn.click(fn=on_load_library, inputs=[library_filter], outputs=[library_output])

        history_btn.click(fn=on_load_history, outputs=[history_output])

    return app

app = build_app()
print("Gradio app built!")

IMPORTANT: You are using gradio version 4.36.1, however version 4.44.1 is available, please upgrade.
--------
Gradio app built!


In [11]:
app.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://6dc0920af4ba314a42.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6dc0920af4ba314a42.gradio.live
